# 5. Évaluation

Il n'existe pas de vérité terrain pour la meilleure rotation. L'évaluation vérifie la cohérence des résultats avec des faits attendus et leur sensibilité aux hypothèses.

In [1]:
import sys
sys.path.insert(0, "..")
import matplotlib.pyplot as plt
from donnees.config import charger
from moteur.series import series_lieux
config = charger()
series = series_lieux(config)
noms = {l["id"]: l["nom"] for l in config["lieux"]}

In [2]:
from moteur.criteres import debuts_par_annee, sensibilite
for lieu, s in series.items():
    debuts = debuts_par_annee(s, config)
    for cle, p in config["priorites"].items():
        a = sensibilite(s, config, debuts, p["poids"])
        print(f"{noms[lieu]:8} {cle:9} stable {a['stable']!s:5}", a["ordres"])

Korhogo  eau       stable False {'reference': [['coton_mais_arachide', 'coton_sorgho_niebe'], ['coton_mais']], 'pluie_reduite': [['coton_sorgho_niebe'], ['coton_mais_arachide'], ['coton_mais']], 'semis_retarde': [['coton_mais_arachide'], ['coton_sorgho_niebe'], ['coton_mais']]}
Korhogo  sol       stable False {'reference': [['coton_mais_arachide', 'coton_sorgho_niebe'], ['coton_mais']], 'pluie_reduite': [['coton_sorgho_niebe'], ['coton_mais_arachide'], ['coton_mais']], 'semis_retarde': [['coton_mais_arachide'], ['coton_sorgho_niebe'], ['coton_mais']]}
Korhogo  debouche  stable False {'reference': [['coton_mais'], ['coton_mais_arachide', 'coton_sorgho_niebe']], 'pluie_reduite': [['coton_mais'], ['coton_sorgho_niebe'], ['coton_mais_arachide']], 'semis_retarde': [['coton_mais'], ['coton_mais_arachide'], ['coton_sorgho_niebe']]}
Bouaké   eau       stable True  {'reference': [['coton_sorgho_niebe'], ['coton_mais_arachide'], ['coton_mais']], 'pluie_reduite': [['coton_sorgho_niebe'], ['coton_

Bouaké   sol       stable True  {'reference': [['coton_sorgho_niebe'], ['coton_mais_arachide'], ['coton_mais']], 'pluie_reduite': [['coton_sorgho_niebe'], ['coton_mais_arachide'], ['coton_mais']], 'semis_retarde': [['coton_sorgho_niebe'], ['coton_mais_arachide'], ['coton_mais']]}


Bouaké   debouche  stable True  {'reference': [['coton_mais'], ['coton_sorgho_niebe'], ['coton_mais_arachide']], 'pluie_reduite': [['coton_mais'], ['coton_sorgho_niebe'], ['coton_mais_arachide']], 'semis_retarde': [['coton_mais'], ['coton_sorgho_niebe'], ['coton_mais_arachide']]}


**Constat**

À Bouaké, les classements résistent à une pluie réduite de 20 % et à un début des pluies retardé de 14 jours. À Korhogo, l'égalité entre les deux rotations à légumineuse se rompt dans les deux scénarios.

**Analyse**

À Korhogo, l'application doit présenter les deux rotations à légumineuse comme équivalentes, et le dit.

In [3]:
from moteur.criteres import couverture
for lieu, s in series.items():
    debuts = debuts_par_annee(s, config)
    sorgho = couverture(s, config["cultures"]["sorgho"], debuts)["part"]
    mais = couverture(s, config["cultures"]["mais"], debuts)["part"]
    print(f"{noms[lieu]:8} couverture sorgho {sorgho:.2f}, maïs {mais:.2f}, sorgho au moins égal {sorgho >= mais}")

Korhogo  couverture sorgho 0.67, maïs 0.50, sorgho au moins égal True
Bouaké   couverture sorgho 0.46, maïs 0.33, sorgho au moins égal True


**Constat**

Dans les deux lieux, le besoin en eau de mi-saison du sorgho est couvert au moins aussi souvent que celui du maïs.

**Analyse**

Ce résultat va dans le sens attendu, mais il vient ici des durées de stades et des coefficients de FAO 56, et non d'une mesure de la tolérance à la sécheresse.

**Limites**

- Le semis est supposé au début des pluies.
- Les valeurs de FAO 56 ne sont pas propres aux variétés ivoiriennes.
- Aucun bilan hydrique du sol ni pluie efficace.
- Les critères sol et débouché sont des hypothèses à confirmer avec l'ANADER.